# 8. Подготовка готовых датасетов для Datalens
 
## Цель: Создать предварительно собранные CSV файлы для прямой загрузки в Datalens

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Настройка путей
project_root = Path.cwd().parent
datalens_path = project_root / 'reports' / 'datalens_ready'
datalens_path.mkdir(parents=True, exist_ok=True)

In [3]:
# Загрузка исходных данных
print("Загрузка данных...")
sales_facts = pd.read_csv(project_root / 'reports' / 'datalens_export' / 'sales_facts.csv')
products_dim = pd.read_csv(project_root / 'reports' / 'datalens_export' / 'products_dimension.csv')
customers_dim = pd.read_csv(project_root / 'reports' / 'datalens_export' / 'customers_dimension.csv')
abc_xyz = pd.read_csv(project_root / 'reports' / 'datalens_export' / 'abc_xyz_matrix.csv')

# Преобразуем даты
sales_facts['date'] = pd.to_datetime(sales_facts['date'])

print(f"✓ Загружено {len(sales_facts)} продаж, {len(products_dim)} товаров, {len(customers_dim)} клиентов")

Загрузка данных...
✓ Загружено 2366 продаж, 859 товаров, 861 клиентов


In [4]:
# 1. ДАТАСЕТ: "Продажи с расширенными метриками"
print("\nСоздание датасета 'Продажи с метриками'...")

sales_metrics = sales_facts.copy()

# Добавляем временные измерения
sales_metrics['year'] = sales_metrics['date'].dt.year
sales_metrics['month'] = sales_metrics['date'].dt.month
sales_metrics['month_name'] = sales_metrics['date'].dt.month_name()
sales_metrics['week'] = sales_metrics['date'].dt.isocalendar().week
sales_metrics['day_of_week'] = sales_metrics['date'].dt.day_name()
sales_metrics['day_of_month'] = sales_metrics['date'].dt.day

# Добавляем информацию о товарах
sales_metrics = sales_metrics.merge(
    products_dim[['sku', 'Бренд', 'Категория', 'Тип', 'current_price']],
    on='sku',
    how='left'
)

# Добавляем информацию о клиентах
sales_metrics = sales_metrics.merge(
    customers_dim[['customer_id', 'segment', 'tier']],
    on='customer_id', 
    how='left'
)

# Расчетные метрики
sales_metrics['avg_price'] = sales_metrics['revenue'] / sales_metrics['quantity']
sales_metrics['profit_margin'] = sales_metrics['profit'] / sales_metrics['revenue']
sales_metrics['profit_margin'] = sales_metrics['profit_margin'].fillna(0)

# Сохраняем
sales_metrics.to_csv(datalens_path / 'sales_metrics.csv', index=False)
print(f"✓ Создан датасет 'sales_metrics.csv' ({len(sales_metrics)} строк)")


Создание датасета 'Продажи с метриками'...
✓ Создан датасет 'sales_metrics.csv' (2366 строк)


In [5]:
# 2. ДАТАСЕТ: "Товары с ABC-XYZ анализом"
print("\nСоздание датасета 'Товары ABC-XYZ'...")

products_abcxyz = products_dim.merge(
    abc_xyz[['sku', 'abc_class', 'xyz_class', 'abc_xyz_class', 'revenue', 'percent_contribution', 'cv']],
    on='sku',
    how='left'
)

# Добавляем стратегические рекомендации
recomendations_map = {
    'AX': '🌟 ЗВЕЗДЫ - Увеличивать продвижение и запасы',
    'AY': '📈 РОСТ - Инвестировать в маркетинг', 
    'AZ': '⚡ ВОПРОСЫ - Изучить нестабильность',
    'BX': '💰 ДОЙНЫЕ КОРОВЫ - Поддерживать наличие',
    'BY': '🔄 СТАБИЛЬНЫЕ - Стандартное управление',
    'BZ': '⚠ РИСКОВАННЫЕ - Контролировать запасы',
    'CX': '📦 МАССОВЫЕ - Минимальные запасы',
    'CY': '🎯 НИШЕВЫЕ - Точечное управление',
    'CZ': '❌ ПРОБЛЕМНЫЕ - Кандидат на исключение'
}

products_abcxyz['strategic_recommendation'] = products_abcxyz['abc_xyz_class'].map(recomendations_map)
products_abcxyz['strategic_recommendation'] = products_abcxyz['strategic_recommendation'].fillna('Не определено')

# Добавляем приоритет
products_abcxyz['priority'] = products_abcxyz['abc_class'].map({'A': 'Высокий', 'B': 'Средний', 'C': 'Низкий'})

products_abcxyz.to_csv(datalens_path / 'products_abcxyz.csv', index=False)
print(f"✓ Создан датасет 'products_abcxyz.csv' ({len(products_abcxyz)} строк)")



Создание датасета 'Товары ABC-XYZ'...
✓ Создан датасет 'products_abcxyz.csv' (859 строк)


In [6]:
# 3. ДАТАСЕТ: "Клиенты с RFM анализом"
print("\nСоздание датасета 'Клиенты RFM'...")

# Сначала агрегируем данные по клиентам из продаж
customer_aggregates = sales_facts.groupby('customer_id').agg({
    'revenue': 'sum',
    'profit': 'sum',
    'order_id': 'nunique',
    'quantity': 'sum',
    'date': ['min', 'max']
}).round(2)

# Упрощаем мультииндекс
customer_aggregates.columns = ['total_revenue', 'total_profit', 'total_orders', 'total_quantity', 'first_order', 'last_order']

customers_rfm = customers_dim.merge(
    customer_aggregates,
    on='customer_id',
    how='left'
)

# Добавляем дополнительные метрики
customers_rfm['avg_order_value'] = customers_rfm['total_revenue'] / customers_rfm['total_orders']
customers_rfm['avg_profit_per_order'] = customers_rfm['total_profit'] / customers_rfm['total_orders']

# Добавляем категории приоритета
segment_priority = {
    'Champions': 'Премиум',
    'Loyal Customers': 'Премиум', 
    'High Value New Customers': 'Растущие',
    'Potential Loyalists': 'Растущие',
    'New Customers': 'Новые',
    'Need Attention': 'Рисковые',
    'About To Sleep': 'Рисковые',
    'At Risk': 'Критические',
    'Cannot Lose Them': 'Критические',
    'High Risk High Value': 'Критические',
    'Low Engagement': 'Низкая активность',
    'Lost Customers': 'Утерянные'
}

customers_rfm['customer_priority'] = customers_rfm['segment'].map(segment_priority)
customers_rfm['customer_priority'] = customers_rfm['customer_priority'].fillna('Другие')

# Рекомендации по работе
engagement_strategy = {
    'Champions': 'Персональные предложения, эксклюзивный доступ',
    'Loyal Customers': 'Программа лояльности, перекрестные продажи',
    'High Value New Customers': 'VIP обслуживание, персональный менеджер',
    'Potential Loyalists': 'Персонализированные рекомендации',
    'New Customers': 'Приветственная программа, обучение',
    'Need Attention': 'Реактивационные кампании, специальные акции',
    'About To Sleep': 'Срочные акции, напоминания',
    'At Risk': 'Восстановительные кампании, опросы',
    'Cannot Lose Them': 'Персональные обращения, специальные условия',
    'High Risk High Value': 'VIP реактивация, анализ поведения',
    'Low Engagement': 'Образовательный контент, стимулы',
    'Lost Customers': 'Анализ причин ухода, специальные условия возврата'
}

customers_rfm['engagement_strategy'] = customers_rfm['segment'].map(engagement_strategy)
customers_rfm['engagement_strategy'] = customers_rfm['engagement_strategy'].fillna('Стандартный маркетинг')

customers_rfm.to_csv(datalens_path / 'customers_rfm.csv', index=False)
print(f"✓ Создан датасет 'customers_rfm.csv' ({len(customers_rfm)} строк)")


Создание датасета 'Клиенты RFM'...
✓ Создан датасет 'customers_rfm.csv' (861 строк)


In [7]:
# 4. ДАТАСЕТ: "Агрегаты по времени"
print("\nСоздание датасета 'Временные агрегаты'...")

# Дневные агрегаты
daily_agg = sales_facts.groupby(pd.Grouper(key='date', freq='D')).agg({
    'revenue': 'sum',
    'profit': 'sum',
    'order_id': 'nunique',
    'customer_id': 'nunique',
    'quantity': 'sum'
}).reset_index()

daily_agg['year'] = daily_agg['date'].dt.year
daily_agg['month'] = daily_agg['date'].dt.month
daily_agg['month_name'] = daily_agg['date'].dt.month_name()
daily_agg['week'] = daily_agg['date'].dt.isocalendar().week
daily_agg['day_of_week'] = daily_agg['date'].dt.day_name()
daily_agg['day_of_week_num'] = daily_agg['date'].dt.dayofweek

daily_agg['avg_order_value'] = daily_agg['revenue'] / daily_agg['order_id']
daily_agg['profit_margin'] = daily_agg['profit'] / daily_agg['revenue']

daily_agg.to_csv(datalens_path / 'daily_aggregates.csv', index=False)
print(f"✓ Создан датасет 'daily_aggregates.csv' ({len(daily_agg)} строк)")



Создание датасета 'Временные агрегаты'...
✓ Создан датасет 'daily_aggregates.csv' (271 строк)


In [8]:
# 5. ДАТАСЕТ: "KPI метрики"
print("\nСоздание датасета 'KPI метрики'...")

# Создаем упрощенный датасет для KPI панели
kpi_data = pd.DataFrame([{
    'total_revenue': sales_facts['revenue'].sum(),
    'total_profit': sales_facts['profit'].sum(),
    'total_orders': sales_facts['order_id'].nunique(),
    'total_customers': sales_facts['customer_id'].nunique(),
    'total_quantity': sales_facts['quantity'].sum(),
    'avg_order_value': sales_facts['revenue'].sum() / sales_facts['order_id'].nunique(),
    'avg_profit_margin': sales_facts['profit'].sum() / sales_facts['revenue'].sum()
}]).round(2)

kpi_data.to_csv(datalens_path / 'kpi_metrics.csv', index=False)
print(f"✓ Создан датасет 'kpi_metrics.csv'")



Создание датасета 'KPI метрики'...
✓ Создан датасет 'kpi_metrics.csv'


In [9]:
# 6. ДАТАСЕТ: "Бренды и категории"
print("\nСоздание датасета 'Бренды и категории'...")

brand_metrics = sales_metrics.groupby('Бренд').agg({
    'revenue': 'sum',
    'profit': 'sum',
    'order_id': 'nunique',
    'customer_id': 'nunique',
    'quantity': 'sum',
    'profit_margin': 'mean'
}).round(2).reset_index()

brand_metrics['avg_revenue_per_order'] = brand_metrics['revenue'] / brand_metrics['order_id']
brand_metrics['revenue_share'] = (brand_metrics['revenue'] / brand_metrics['revenue'].sum() * 100).round(1)

category_metrics = sales_metrics.groupby('Категория').agg({
    'revenue': 'sum',
    'profit': 'sum', 
    'order_id': 'nunique',
    'customer_id': 'nunique',
    'quantity': 'sum',
    'profit_margin': 'mean'
}).round(2).reset_index()

category_metrics['avg_revenue_per_order'] = category_metrics['revenue'] / category_metrics['order_id']
category_metrics['revenue_share'] = (category_metrics['revenue'] / category_metrics['revenue'].sum() * 100).round(1)

brand_metrics.to_csv(datalens_path / 'brand_metrics.csv', index=False)
category_metrics.to_csv(datalens_path / 'category_metrics.csv', index=False)
print(f"✓ Созданы датасеты 'brand_metrics.csv' ({len(brand_metrics)} брендов) и 'category_metrics.csv' ({len(category_metrics)} категорий)")


Создание датасета 'Бренды и категории'...
✓ Созданы датасеты 'brand_metrics.csv' (11 брендов) и 'category_metrics.csv' (4 категорий)
